# Lezione 8 — Distribuzione: TFLite e TensorFlow.js

Corso **"Fondamenti di TensorFlow da zero"** — DIH DATAlife

## Obiettivi di questa lezione

- Convertire un modello TensorFlow in TFLite (TensorFlow Lite) per dispositivi mobili
- Verificare che il modello convertito funzioni su un dispositivo fisico o virtuale
- Esportare un modello TensorFlow in TensorFlow.js per eseguirlo direttamente nel browser
- Capire i compromessi tra precisione, dimensione e velocità nella distribuzione

## 1. Preparare i dati e verificare la versione di TensorFlow

Usiamo Fashion MNIST: il dataset di 70.000 immagini di abbigliamento che abbiamo visto prima. In questo capitolo lo prepariamo, addestriamo un modello e lo convertiamo in due formati di distribuzione.

In [ ]:
import tensorflow as tf
import numpy as np
import os
import time

print("Versión de TensorFlow:", tf.__version__)

In [ ]:
# Cargamos Fashion MNIST directamente desde Keras
fashion_mnist = tf.keras.datasets.fashion_mnist
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# Normalizamos los valores de los píxeles al rango 0-1
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

nombres_clases = [
    "Camiseta", "Pantalón", "Jersey", "Vestido", "Abrigo",
    "Sandalia", "Camisa", "Zapatilla", "Bolso", "Botín"
]

print("Forma de x_train:", x_train.shape)
print("Forma de x_test:", x_test.shape)

## 2. Addestrare un modello piccolo con Keras

Il modello è deliberatamente semplice e piccolo (una rete densa con due livelli): vogliamo vedere come ridurre ulteriormente le dimensioni e la latenza, che sono le priorità chiave per i dispositivi mobili.

In [ ]:
modelo = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])

modelo.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

modelo.summary()

In [ ]:
# Entrenamos pocas épocas: el objetivo de este notebook es el despliegue, no la precisión
modelo.fit(x_train, y_train, epochs=5, validation_split=0.1)

In [ ]:
perdida_test, precision_test = modelo.evaluate(x_test, y_test, verbose=0)
print(f"Precisión sobre el conjunto de test: {precision_test:.4f}")

## 3. Convertire il modello a TensorFlow Lite

Con il modello già addestrato, lo convertiamo in TensorFlow Lite. TFLite non è solo una copia—è una versione ottimizzata, spesso quantizzata.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(modelo)
modelo_tflite = converter.convert()

ruta_tflite = "modelo_fashion_mnist.tflite"
with open(ruta_tflite, "wb") as f:
    f.write(modelo_tflite)

print("Modelo convertido y guardado en", ruta_tflite)

## 4. Controllare la dimensione del file su disco

Uno dei vantaggi di TensorFlow Lite è la riduzione della dimensione. Il modello salvato come `.keras` è più grande; la versione TFLite è molto più piccola.

In [ ]:
# Guardamos también el modelo original de Keras para comparar tamaños
ruta_keras = "modelo_fashion_mnist.h5"
modelo.save(ruta_keras)

tamano_tflite_kb = os.path.getsize(ruta_tflite) / 1024
tamano_keras_kb = os.path.getsize(ruta_keras) / 1024

print(f"Tamaño del modelo TFLite:  {tamano_tflite_kb:.1f} KB")
print(f"Tamaño del modelo Keras (.h5): {tamano_keras_kb:.1f} KB")

## 5. Eseguire un'inferenza di prova con l'interprete di TFLite

L'interprete di TFLite è lo strumento che esegue il modello su un dispositivo mobile. È scritto in C++ ed è estremamente leggero.

In [ ]:
interpreter = tf.lite.Interpreter(model_path=ruta_tflite)
interpreter.allocate_tensors()

detalles_entrada = interpreter.get_input_details()
detalles_salida = interpreter.get_output_details()

print("Forma de entrada esperada:", detalles_entrada[0]["shape"])
print("Forma de salida esperada:", detalles_salida[0]["shape"])

In [ ]:
# Tomamos una imagen de test y le añadimos la dimensión de lote (batch) que espera el modelo
imagen_prueba = x_test[0:1].astype("float32")

interpreter.set_tensor(detalles_entrada[0]["index"], imagen_prueba)
interpreter.invoke()
prediccion_tflite = interpreter.get_tensor(detalles_salida[0]["index"])

clase_predicha = np.argmax(prediccion_tflite[0])
print("Predicción del modelo TFLite:", nombres_clases[clase_predicha])
print("Etiqueta real:", nombres_clases[y_test[0]])

## 6. Confrontare il tempo di inferenza: Keras contro TFLite

Per darti un'idea della velocità, misuriamo il tempo di inferenza su una singola immagine con Keras e con TFLite.

In [ ]:
num_muestras = 200
lote_prueba = x_test[:num_muestras].astype("float32")

# Tiempo del modelo Keras original
inicio = time.time()
modelo.predict(lote_prueba, verbose=0)
tiempo_keras = time.time() - inicio

# Tiempo del modelo TFLite, imagen a imagen (así es como se usa en un dispositivo real)
inicio = time.time()
for i in range(num_muestras):
    interpreter.set_tensor(detalles_entrada[0]["index"], lote_prueba[i:i+1])
    interpreter.invoke()
    _ = interpreter.get_tensor(detalles_salida[0]["index"])
tiempo_tflite = time.time() - inicio

print(f"Tiempo Keras  ({num_muestras} imágenes): {tiempo_keras:.4f} s")
print(f"Tiempo TFLite ({num_muestras} imágenes): {tiempo_tflite:.4f} s")

I tempi concreti dipendono dalla macchina dove esegui il notebook (CPU del tuo computer, o GPU di Colab). L'importante è che TFLite è generalmente più veloce, soprattutto su hardware a bassa potenza (smartphone).

## 7. TensorFlow.js: portare il modello nel browser (non eseguibile in questo notebook)